#### 1. Setup and Libraries

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

intl_oem = ['Grundfos (Denmark)', 'Pedrollo (Italy)', 'SolarWorld (Germany)', 'Speroni (Italy)']
house_brand = ['Dayliff (Global)']
local_vendors = ['Ethio-Plastics', 'Addis Engineering', 'Nile Logistics']

product_families = ['Surface Pumps', 'Borehole Pumps', 'Solar Power Systems', 'Water Treatment', 'Pool Equipment']
num_orders = 300

#### 2. The Base Data Generation

In [2]:
data = []

for i in range(num_orders):
    rand = np.random.random()
    if rand < 0.4: 
        supplier = 'Dayliff (Global)'
        category = 'Internal Brand'
    elif rand < 0.8: 
        supplier = np.random.choice(intl_oem)
        category = 'External OEM'
    else: 
        supplier = np.random.choice(local_vendors)
        category = 'Local Partner'

    po_date = datetime(2024, 1, 1) + timedelta(days=np.random.randint(0, 700))
    
    if category == 'Internal Brand':
        lead_time = np.random.randint(20, 30)
    elif category == 'External OEM':
        lead_time = np.random.randint(35, 60)
    else:
        lead_time = np.random.randint(3, 12)
        
    expected_delivery = po_date + timedelta(days=lead_time)
    
    # Financials
    price_po = np.random.uniform(200, 1500)
    qty = np.random.randint(10, 200)

    data.append({
        'PO_Number': f"DS-2025-{1000+i}",
        'Supplier': supplier,
        'Category': category,
        'Product_Family': np.random.choice(product_families),
        'PO_Date': po_date,
        'Expected_Delivery': expected_delivery,
        'Actual_Delivery': expected_delivery, 
        'Ordered_Qty': qty,
        'Received_Qty': qty,       
        'Unit_Price_PO': round(price_po, 2),
        'Unit_Price_Final': round(price_po, 2)
    })

df = pd.DataFrame(data)

#### 3. Injecting the "Deep Look" Anomalies

In [3]:
# 1. OEM Price Creep (Simulating hidden shipping/import costs)
oem_idx = df[df['Category'] == 'External OEM'].sample(frac=0.15).index
df.loc[oem_idx, 'Unit_Price_Final'] *= 1.10 

# 2. Local Supply Chain Latency
local_idx = df[df['Category'] == 'Local Partner'].sample(frac=0.20).index
df.loc[local_idx, 'Actual_Delivery'] += timedelta(days=np.random.randint(5, 15))

# 3. Major OEM Delay (Simulating a 2024 Shipping Crisis)
delay_idx = df[df['Supplier'] == 'Grundfos (Denmark)'].sample(frac=0.10).index
df.loc[delay_idx, 'Actual_Delivery'] += timedelta(days=30)

# 4. Data Entry Error (Negative Lead Time - Impossible!)
df.loc[5, 'Actual_Delivery'] = df.loc[5, 'PO_Date'] - timedelta(days=5)

df.to_csv('../data/raw/procurement_raw.csv', index=False)
print("Strategic Dataset Created (2024-2025).")

Strategic Dataset Created (2024-2025).


In [5]:
df.tail()

,PO_Number,Supplier,Category,Product_Family,PO_Date,Expected_Delivery,Actual_Delivery,Ordered_Qty,Received_Qty,Unit_Price_PO,Unit_Price_Final
295,DS-2025-1295,Nile Logistics,Local Partner,Borehole Pumps,2024-07-20,2024-07-31,2024-08-08,173,173,324.20,324.20
296,DS-2025-1296,Pedrollo (Italy),External OEM,Borehole Pumps,2025-08-03,2025-09-21,2025-09-21,117,117,505.81,505.81
297,DS-2025-1297,Grundfos (Denmark),External OEM,Water Treatment,2025-10-23,2025-12-12,2025-12-12,172,172,605.95,605.95
298,DS-2025-1298,Dayliff (Global),Internal Brand,Pool Equipment,2025-06-06,2025-07-05,2025-07-05,86,86,533.95,533.95
299,DS-2025-1299,Dayliff (Global),Internal Brand,Solar Power Systems,2024-11-15,2024-12-07,2024-12-07,155,155,976.16,976.16
